# 17 — Train the motion and structure JEPA grid on real GAVD

This is the training continuation of notebooks 15 and 16. The default
data path uses the same **625 GAVD clips, 93 source videos, five outer
folds and seeds 42--46** as the existing real-data investigations.
Every experiment is repeated over all 25 train/test combinations.

Set `RUN_TRAINING = True` in the configuration cell to execute the grid,
or set `LATERALITY_RESEARCH_RUN_REAL=1` before starting the kernel, as in
Notebook 12. With the switch off, Run All still prepares/verifies real
inputs, displays the full workload and checks saved-job availability.
This keeps the expensive training decision visible before optimization.
Saved outputs are consumed directly by Notebook 18 in a fresh kernel.

### What to look for in the reviewed results — 2026-09-08

The full motion/region grid has completed: Notebook 18 and its saved
artifacts verify 50 paired jobs, 125 trained encoders and 150,000 updates.
This copy of Notebook 17 had no saved code outputs when reviewed, so
Section 6 uses explicitly identified artifacts to interpret training.
Falling prediction loss and useful movement readout give different answers
here. The completed grid used **CUDA BF16 training and FP32 evaluation**;
that setting matters when reopening it from the configuration below.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

def locate_suite():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "neurips-laterality"):
            if (candidate / "laterality_extensions/motion_structured_masks.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Run from the research project directory.")

SUITE_ROOT = locate_suite()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))
from laterality_extensions.motion_structured_masks import (
    StudyArm, mamp_logits, sample_study_mask, study_arms,
    paired_study_masks, context_cue_audit,
)
from laterality_extensions.comparative_masks import motion_scores
from notebook_progress import (
    NotebookTaskProgress, run_notebook_task, study_inputs_with_progress,
    audit_training_masks_with_progress, grid_status_with_progress,
    run_gavd_grid_with_progress, collect_gavd_grid_with_progress,
    evaluate_retained_motion_with_progress,
)
set_matplotlib_formats("svg", "png")
pd.set_option("display.precision", 3)

## 1. Load the real GAVD cohort and declare the full split/seed grid

Run cells in order in a Python kernel with the project's dependencies.
Long tasks use the shared `notebook_progress.py` wrapper: one updating
display shows the current stage, fold/seed, elapsed time and estimated
remaining time. Mask batches and optimizer updates appear within their
active stage. ETA adjusts as stages finish; their costs differ. Cached,
disabled, missing-input and failed tasks receive explicit status labels.
`DATA_MODE="gavd"` is the default. The helper below follows the same
preparation and source splitting functions as notebooks 01 and 02:

1. Verify an existing paper-profile cohort and split manifest by their
   content hashes. If absent, read the local GAVD pose archives and
   official annotations, apply the existing QC and target rules, and
   create those two artifacts. The first run takes longer.
2. The protocol fixes 642 pose archives and 666 annotations. If the local
   cache contains later additions, recover the original extraction
   generations only when their inventory **exactly** matches the locked
   count and SHA-256. Store verified copies under the paper artifact
   root. The original cache stays intact. A mismatch stops with an
   actionable error; it cannot silently switch to generated data.
3. The reference QC result is 625 clips from 93 source videos. Inputs have
   shape `[clips, 64, 33, 3]`; four prepared steps form each of 16 tokens
   per landmark. Naturally missing observations remain marked invalid.
   The target is a coordinate-derived bilateral movement contrast, not
   the dataset's condition annotation.
4. Reuse the five video-disjoint outer folds. Seeds 42--46 change
   initialization, source draws, augmentations and masks; they repeat
   the **same** train/test partitions. A video is held out exactly once
   per seed. There are 25 fold/seed combinations, not 25 independent
   datasets. Video separation does not establish subject separation.

The printed census makes train and test membership visible. Source IDs
and sequence IDs are retained in `inputs["memberships"]`. The reference
train/test clip counts are 436/189, 443/182, 553/72, 548/77 and 520/105.
Unequal clip counts are expected because entire videos stay together.

A small generated-data path remains available only through the explicit
`LATERALITY_MOTION_DATA_MODE=synthetic` software-check setting. It prints
its reduced scope and uses a separate artifact directory. It provides
no GAVD results. See [the run guide](docs/MOTION_GAVD_WORKFLOW.md) for
paths, environment settings, recovery and a notebook-by-notebook walkthrough.

In [ ]:
from laterality_extensions.motion_gavd import gavd_plan, readout_contrasts
DATA_MODE = os.getenv("LATERALITY_MOTION_DATA_MODE", "gavd")
FOLDS = (0, 1, 2, 3, 4)
SEEDS = (42, 43, 44, 45, 46)
EXPERIMENTS = tuple(os.getenv("LATERALITY_MOTION_EXPERIMENTS", "motion,regions").split(","))
CREATE_MISSING_INPUTS = True
DEVICE = os.getenv("LATERALITY_DEVICE", "auto")
# Keep the numerical mode identical in Notebooks 17 and 18.
PRECISION = os.getenv("LATERALITY_MOTION_PRECISION", "fp32")  # or "bf16" on native CUDA BF16 hardware
RESUME_INTERVAL = int(os.getenv("LATERALITY_MOTION_RESUME_INTERVAL", "100"))
# Same explicit training switch as Notebook 12; also accept the study-specific alias.
RUN_TRAINING = os.getenv("LATERALITY_MOTION_RUN_REAL",
                        os.getenv("LATERALITY_RESEARCH_RUN_REAL", "0")) == "1"
if DATA_MODE == "synthetic":
    FOLDS, SEEDS = (0,), (42,)
    print("EXPLICIT SYNTHETIC SOFTWARE CHECK: one fold/seed, one update, no GAVD evidence")
OUTPUT_ROOT = Path(os.getenv("LATERALITY_MOTION_OUTPUT_ROOT", str(SUITE_ROOT / "artifacts" /
    ("motion_structured" if DATA_MODE == "gavd" else "motion_structured_synthetic"))))
print(f"Mode={DATA_MODE}; folds={FOLDS}; seeds={SEEDS}; device={DEVICE}")
print(f"Training enabled={RUN_TRAINING}; outputs={OUTPUT_ROOT}")

In [ ]:
input_progress = NotebookTaskProgress("Dataset preparation and source splits", "stage")
inputs = study_inputs_with_progress(mode=DATA_MODE, folds=FOLDS, seeds=SEEDS,
    create_missing=CREATE_MISSING_INPUTS, progress=input_progress)
display(inputs["census"])
assert inputs["census"].source_overlap.eq(0).all()
display(inputs["memberships"].head(8))
if DATA_MODE == "gavd":
    cohort = inputs["cohort"]
    display(cohort.table.groupby("condition").agg(
        accepted_clips=("sequence_id", "size"), source_videos=("video_id", "nunique")))
    display(pd.Series({key: cohort.attrition[key] for key in
        ("input_sequences", "accepted_sequences", "accepted_sources", "excluded_sequences")}))
    print("Cohort:", cohort.cohort_digest)
    print("Split:", inputs["splits"]["split_digest"])
    print("Artifacts:", inputs["context"].artifact_root)

## 2. Freeze the questions and training recipe

| Experiment | Paired arms | Real grid |
|---|---|---:|
| Motion | Uniform, MAMP code convention, robust motion mixture | 3 × 5 folds × 5 seeds = 75 encoders |
| Regions | Connected six-landmark half-window region, count-matched uniform | 2 × 5 folds × 5 seeds = 50 encoders |

Together these are **50 paired jobs, 125 encoders and 150,000 optimizer
updates**. An update trains every arm in its job on identical source
draws and geometric views. The two uniform controls have different
target budgets and remain separate. To conduct the declared trajectory
or completion follow-up, add its experiment name here and in Notebook 18.
A fold/seed subset is a pilot and is labeled as such.

Like Notebook 12, use the starting recipe retained from Notebook 08:
1,200 updates, batch 20, width 96, four encoder layers, two predictor
layers and four attention heads. The tracked summary provides the
settings without requiring old local checkpoints. Read its complete
configuration below; do not inherit tiny teaching-model defaults.

In [ ]:
plan = gavd_plan(inputs, experiments=EXPERIMENTS, device=DEVICE, output_dir=OUTPUT_ROOT,
                 precision=PRECISION, resume_interval=RESUME_INTERVAL)
display(pd.Series({key: plan[key] for key in
    ("scope", "training_runs", "optimizer_updates", "recipe_source", "output_dir")}))
display(pd.Series(plan["settings"], name="Declared training settings"))
display(pd.Series(plan["execution"], name="Execution and recovery settings"))
display(plan["workload"].groupby(["experiment", "fold", "seed"], sort=False).agg(
    encoders=("condition", "size"), optimizer_updates=("updates", "sum")))
display(pd.DataFrame([{"experiment": e, **arm} for e, arms in plan["arms"].items()
                      for arm in arms.values()]))

### Hardware, numerical, and cache preflight

The requested device is only a preference until the **current notebook
kernel's PyTorch build** resolves it. `DEVICE="auto"` chooses CUDA when
this kernel has CUDA support, Apple MPS when available, and otherwise
CPU. Seeing an NVIDIA adapter in Task Manager or `nvidia-smi` is not
sufficient: a CPU-only PyTorch wheel still resolves `auto` to CPU. The
next cell reports both views of the machine. If it detects NVIDIA but
`torch.cuda.is_available()` is false, install a CUDA-enabled PyTorch
build in the environment used by this kernel, restart the kernel, and
rerun from the configuration cell. An explicit `DEVICE="cuda"` fails
instead of silently falling back.

**Windows setup for this workspace:** the review found an RTX 4090
Laptop GPU (16 GiB) behind a CPU-only PyTorch kernel. A separate kernel,
**GAVD5 CUDA (PyTorch 2.13)**, was installed and verified with an actual
CUDA matrix multiplication. Select it in the notebook's kernel picker,
then run from the top. To recreate it from the repository root:

```powershell
.\neurips-laterality\scripts\setup_cuda_kernel.ps1
```

The script uses the official PyTorch CUDA 13.0 wheel and preserves the
existing environment's other package versions. It does not restart your
running kernels. Real training with `DEVICE="auto"` now stops when it
detects NVIDIA hardware but cannot use CUDA; an intentional CPU pilot
must use `DEVICE="cpu"` explicitly.

Real training keeps only the current outer-training fold, its validated
target masks, and its sampling schedule resident on the selected device;
outer-test tensors remain sealed. Shared geometric views are generated
on that device, and CUDA uses fused AdamW. Jobs run serially on one GPU
so independent fold/seed models do not contend for memory.

Content-keyed motion scores are reused across clip draws and seeds.
These deterministic scores never include target labels. Every stochastic
mask still uses its original seed, step and clip-offset stream. Fixed-size
target gathers avoid CUDA synchronization from boolean indexing, and
multi-tensor teacher EMA updates reduce small kernel launches. The grid
reuses the training tensor bank until the outer fold changes.

`PRECISION="fp32"` retains the reference numerical mode. For faster CUDA
training, set **`PRECISION="bf16"`** in the configuration cell and use
the same value in Notebook 18. Native BF16 support is checked before
training. Transformer/projector matrix operations use autocast while
model weights, AdamW states, teacher EMA, centers, sharpened softmax,
variance/covariance losses and frozen evaluation remain FP32. BF16 has
a **different training/cache identity**; its results must be reported as
a separate numerical mode. A finite pilot does not establish identical
long-run convergence.

FP32 matrix reductions explicitly use the highest-precision policy;
the caller's previous setting is restored afterward. `torch.compile`
stays disabled: the native Windows environment and this small model
need a separate compilation benchmark before paying compilation costs.
PyTorch selects a compatible attention backend with the validity masks
intact. The measured BF16 run used memory-efficient CUDA attention and
Tensor Core GEMMs. Do not remove missing-token masks to force FlashAttention.

FP8, INT8 and 8-bit AdamW are deliberately absent from the configuration.
This Ada GPU has FP8-capable Tensor Cores, but the native-Windows CUDA
environment has no FP8 training package, and FP8 needs explicit scaling
state, compatible layers, resume support and a separate validation grid.
Eight-bit Adam would save only about 12 MiB across the three motion arms
against roughly 684 MiB measured BF16 peak allocation. INT8 is relevant
to a later deployment benchmark, not this gradient-based pretraining or
FP32 frozen-feature evaluation. See the performance review's
[precision decision](docs/MOTION_PRETRAINING_PERFORMANCE.md#fp8-and-8-bit-quantization-decision).

There are three distinct kinds of reuse:

| Layer | What it saves | When it may count |
|---|---|---|
| Complete training cache | All encoder arms, histories, schedules, and controls | Only after identity, inventory, hashes, shapes, and finite values validate |
| Fold tensors and clip scores | Host-to-device copies and repeated motion medians | In memory, within the current training fold; content changes invalidate scores |
| Paired resume candidate | Model, projector, optimizer, and history at one shared update boundary | Only when periodic resume is enabled and its checksum and full state validate; never by itself a completed result |
| Evaluation/grid cache | Frozen features, readouts, diagnostics, and pooled tables | Only after its training identity and table contents validate |

`RESUME_INTERVAL=100` is connected to every real training job by default.
A shared boundary stores every arm's model, projector, optimizer and
history, with a checksum. At most the updates after the last saved
boundary need repeating. Final checkpoints remain separate. Set zero
to disable periodic recovery; it does not change the numerical identity.
Per-job readout caches store reduced tables; resident evaluation inputs
and features are transient. Identical initial/direct-pose readouts are
fitted once per job, and a validated pooled report reuses its bootstrap.

The inventory below prints the exact expected paths. A cache candidate's
mere existence is never evidence. Changing data, folds, seeds, model or
mask code, backend, PyTorch runtime, or numerical mode intentionally
produces a different content identity rather than overwriting old work.
Explicit synthetic mode remains a separate **one-fold, one-seed,
one-update CPU software check** and produces no GAVD evidence.

In [ ]:
import shutil
import subprocess
import torch
from laterality_extensions.masked_learning import configure_learning_runtime
from laterality_extensions.motion_runtime import motion_numerical_policy

def visible_nvidia_adapters():
    """Report NVIDIA hardware independently of PyTorch, without a shell."""
    executable = shutil.which("nvidia-smi")
    if executable is None:
        return pd.DataFrame(columns=["adapter", "memory_mib", "driver", "compute_capability"])
    command = [executable,
        "--query-gpu=name,memory.total,driver_version,compute_cap",
        "--format=csv,noheader,nounits"]
    try:
        completed = subprocess.run(command, capture_output=True, text=True,
            timeout=10, check=True,
            creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
    except (OSError, subprocess.SubprocessError):
        return pd.DataFrame(columns=["adapter", "memory_mib", "driver", "compute_capability"])
    rows = []
    for line in completed.stdout.splitlines():
        values = [value.strip() for value in line.split(",")]
        if len(values) == 4:
            rows.append(dict(zip(
                ("adapter", "memory_mib", "driver", "compute_capability"), values)))
    return pd.DataFrame(rows)

# Synthetic plans deliberately replace any requested accelerator with CPU.
effective_request = str(plan["settings"]["device"])
runtime_error = None
try:
    resolved_runtime = configure_learning_runtime(effective_request)
except (RuntimeError, ValueError) as error:
    runtime_error = error
    resolved_runtime = {"device": "unavailable", "cpu_threads": None, "accelerated": False}

nvidia_adapters = visible_nvidia_adapters()
torch_cuda_ready = bool(torch.cuda.is_available())
if not nvidia_adapters.empty and not torch_cuda_ready:
    accelerator_note = (
        "NVIDIA hardware is visible, but this kernel's PyTorch build cannot use CUDA. "
        "Install a CUDA-enabled PyTorch build in this kernel environment and restart."
    )
elif torch_cuda_ready:
    accelerator_note = "CUDA is available to this notebook kernel."
elif resolved_runtime["device"] == "mps":
    accelerator_note = "Apple MPS is available to this notebook kernel."
else:
    accelerator_note = "This plan will use the CPU."

runtime_card = pd.DataFrame([
    {"check": "data mode", "value": DATA_MODE},
    {"check": "user-requested device", "value": DEVICE},
    {"check": "effective plan request", "value": effective_request},
    {"check": "resolved training device", "value": resolved_runtime["device"]},
    {"check": "PyTorch", "value": torch.__version__},
    {"check": "PyTorch CUDA runtime", "value": torch.version.cuda or "none (CPU-only build)"},
    {"check": "cuDNN", "value": torch.backends.cudnn.version() or "unavailable"},
    {"check": "FP32 matmul policy", "value": torch.get_float32_matmul_precision()},
    {"check": "training precision", "value": plan["execution"]["precision"]},
    {"check": "periodic paired resume", "value": plan["execution"]["resume_interval"]},
    {"check": "kernel executable", "value": sys.executable},
    {"check": "torch.compile", "value": "disabled by this workflow"},
    {"check": "real training enabled", "value": RUN_TRAINING},
    {"check": "readiness", "value": accelerator_note},
])
display(runtime_card.style.hide(axis="index"))
if not nvidia_adapters.empty:
    display(nvidia_adapters.style.hide(axis="index").set_caption(
        "NVIDIA adapters visible to the operating system"))
if runtime_error is not None:
    raise RuntimeError(
        "CUDA preflight failed before training. "
        f"Requested device: {effective_request!r}. {runtime_error} "
        "A running notebook cannot change its Python environment: select "
        "'GAVD5 CUDA (PyTorch 2.13)' in the kernel picker, restart the kernel, "
        "and Run All from the top."
    ) from runtime_error
print("Effective numerical contract:", motion_numerical_policy(
    resolved_runtime["device"], plan["execution"]["precision"]))

### Measure a bounded pilot before the complete grid

The optional cell runs **12 paired motion updates on real fold 0** using
batch 20, width 96 and the declared four/two-layer model. It excludes
warmup and the final weight-copy boundary from the steady-state timing.
It writes a timing report, not a scientific checkpoint. Keep it disabled
during routine Run All, or set `RUN_BENCHMARK=True` to inspect throughput
and peak allocated GPU memory through the same progress wrapper.

On this laptop, short measurements gave about 3.85 seconds per paired
CPU update, 0.177 seconds with optimized CUDA FP32 and 0.096 seconds with
CUDA BF16. These are throughput checks, not full-grid runtime promises;
preparation, checkpoint writes, frozen readouts and CPU ridge fitting
add time. The full mask schedule also occupies memory beyond this short
pilot. See [the performance review](docs/MOTION_PRETRAINING_PERFORMANCE.md)
for the measurements, profiler evidence and reproduction commands.

Keep the scientific batch size at 20. A larger batch changes the number
of clips seen and the VICReg covariance estimate. GPU memory occupancy
alone does not measure speed. For this resident dataset, adding a
DataLoader, pinned-memory workers or a second transfer stream would add
machinery to an optimization loop that already needs no input transfer.

In [ ]:
from notebook_progress import run_notebook_task
from scripts.benchmark_motion_pretraining import benchmark
RUN_BENCHMARK = os.getenv("LATERALITY_MOTION_BENCHMARK", "0") == "1"
benchmark_progress = NotebookTaskProgress("Real GAVD throughput pilot", "stage")
timing_report = run_notebook_task(benchmark, progress=benchmark_progress,
    label="Twelve paired motion updates; warmup excluded from throughput",
    enabled=RUN_BENCHMARK and DATA_MODE == "gavd",
    device=resolved_runtime["device"], precision=plan["execution"]["precision"],
    steps=12, output=OUTPUT_ROOT / "performance" / f"{plan['execution']['precision']}.json")
if timing_report is not None:
    display(pd.Series({key: value for key, value in timing_report.items()
                       if key not in ("identity", "history")}))

In [ ]:
workload = plan["workload"].groupby(["experiment", "fold"], sort=False).size().unstack(0)
ax = workload.plot.bar(figsize=(8, 3.5), title=f"{DATA_MODE.upper()}: encoders across declared seeds")
ax.set(xlabel="Outer fold", ylabel="Encoders")
ax.figure.tight_layout(); display(ax.figure); plt.close(ax.figure)
checkpoint_progress = NotebookTaskProgress("Training checkpoint inspection", "stage")
jobs_before = grid_status_with_progress(plan, inputs, progress=checkpoint_progress)
display(jobs_before)

from laterality_extensions.comparative_training import _resume_checksum_path, _resume_path
cache_rows = []
for job in jobs_before.itertuples(index=False):
    training_directory = Path(job.training_directory)
    resume_path = _resume_path(training_directory)
    checksum_path = _resume_checksum_path(resume_path)
    resume_files = (resume_path.is_file(), checksum_path.is_file())
    resume_state = (
        "paired candidate present; validate before resume"
        if all(resume_files) else
        "incomplete candidate; fail closed" if any(resume_files) else
        "absent"
    )
    cache_rows.append({
        "experiment": job.experiment,
        "fold": job.fold,
        "seed": job.seed,
        "complete_cache": (
            "validated complete" if str(job.training_status).startswith("complete") else "missing"
        ),
        "resume_cache": resume_state,
        "training_directory": str(training_directory),
        "resume_path": str(resume_path),
        "resume_checksum_path": str(checksum_path),
    })
cache_inventory = pd.DataFrame(cache_rows)
cache_summary = (cache_inventory.groupby(
    ["complete_cache", "resume_cache"], dropna=False).size()
    .rename("paired_jobs").reset_index())
display(cache_summary)
display(cache_inventory)
print("Exact cache inventory is retained in `cache_inventory`; output root:",
      Path(plan["output_dir"]).resolve())

## 3. Trace one fold through the information boundaries

1. Take only the current fold's training videos for self-supervised
   optimization. Sample videos uniformly and clips within videos; many
   clips from one recording must not dominate the source schedule.
2. Initialize the online encoder, predictor and regularizer projector
   once per job. Copy their parameters into every mask arm. Seeds are
   explicit, and each fold/seed starts a new model.
3. Use separate streams for clip exposure, shared geometric views and
   arm-specific mask draws. Validate every scheduled mask before the
   first optimizer update. The mask sampler never reads the target label.
4. Feed visible context to the online encoder/predictor and full valid
   input to the teacher. Optimize centered teacher-feature cross-entropy
   plus the shared unmasked feature-variation regularizer. Average target
   loss within each clip, then average clips, even with ragged masks.
5. Keep the teacher gradient-free; update its parameters by EMA after
   each online optimizer step. Save online, teacher, predictor, target
   center, initial features, schedules, policies and training histories.
6. Freeze the encoder. Fit preprocessing and ridge readout using only
   training sources, then predict every outer-test clip once for this
   seed/arm. Test outcomes cannot choose masks, checkpoints or penalties.

The objective retains AdamW betas (0.9, 0.95), constant learning rate,
temperatures 0.06/0.10, center momentum 0.9, gradient clipping at 1,
rotations up to eight degrees and translations up to 0.03 prepared units.
All arms retain the same 33-landmark input, twelve-landmark regularizer
pool and five bilateral readout pairs. These remaining anatomical choices
are part of the gait adaptation, not learned discoveries.

In [ ]:
from laterality_extensions.masked_learning import LearningSettings
first_fold, first_seed = FOLDS[0], SEEDS[0]
data = inputs["datasets"][first_fold]
local = replace(LearningSettings(**plan["settings"]), fold=first_fold, seed=first_seed)
rows = data.train_rows[:local.batch_size]
display(pd.DataFrame({"sequence_id": data.sequence_ids[rows], "source_id": data.source_ids[rows], "role": "train"}))
preview = []
for experiment in EXPERIMENTS:
    arms = {name: StudyArm(**spec) for name, spec in plan["arms"][experiment].items()}
    masks, _ = paired_study_masks(data, rows, local, arms)
    for name, mask in masks.items():
        preview.append({"experiment": experiment, "condition": name,
            "fold": first_fold, "seed": first_seed,
            "smallest_target_count": int(mask.sum((1, 2)).min()),
            "largest_target_count": int(mask.sum((1, 2)).max())})
display(pd.DataFrame(preview))
print("Preview only. The training runner validates the complete sampled schedule for every job.")

## 4. Execute or reuse the complete declared grid

Confirm the printed mode, folds, seeds, arm definitions, settings and
output directory, then enable the configuration switch to train.
The updating progress display shows the fold, seed, experiment, mask
preflight and optimizer update count. It then identifies frozen-feature
encoding, ridge fitting, predictor diagnostics and cached-table checks.
Training-only tensors and masks remain resident, shared augmentations are
generated on the selected CPU/CUDA/MPS device, and accelerator diagnostics
are transferred back only at bounded reporting or checkpoint boundaries.
Runtime depends on the backend; the hardware card records what this kernel
can actually use rather than assuming that `DEVICE="auto"` means GPU.

A complete compatible job is reused by content identity. When periodic
paired resume is configured, an interrupted job may continue only from a
checksum-validated shared-arm optimizer boundary; otherwise it restarts
from its seed. A resume or incomplete staging directory is not completed
evidence. Per-job
readout tables are cached separately, so changing evaluation can reuse
trained encoders when their training identity is unchanged.

The default artifact root is `artifacts/motion_structured`. Each training
job has a manifest and checkpoint files under its experiment/digest.
Each evaluation has its own digest under `evaluations`. A complete grid
adds a table index, source membership and pooled summaries under `grids`.

In [ ]:
# Synthetic mode is an explicitly requested one-update software check.
training_progress = NotebookTaskProgress("Motion and structure JEPA grid", "stage")
result = run_gavd_grid_with_progress(plan, inputs, progress=training_progress,
    enabled=RUN_TRAINING or DATA_MODE == "synthetic")
print(result["status"])
display(result["jobs"])
if result["status"] == "Complete":
    print("Complete grid report:", result["directory"])
    display(result["per_seed"][["experiment", "condition", "representation", "seed",
        "r2", "mae", "evaluated_clips", "evaluated_sources"]])
else:
    print("Enable RUN_TRAINING in the configuration cell, rerun that cell, then this cell.")

## 5. Interpret the outcome at the correct level

Finite losses show optimization is functioning. An arm's own teacher
changes during learning, so a lower own-teacher loss cannot rank useful
movement information across arms. Notebook 18 evaluates trained online,
EMA teacher and initial encoders with the same frozen readouts, alongside
direct-pose features and the training mean.

It pools all five held-out folds **within each seed** before computing
source-balanced R² and MAE. Fold R² values must not be averaged. Every
video receives equal total weight, regardless of its clip count. Seed
variation reflects training randomness on reused data, not extra subjects.

A motion-sensitive readout that helps trained and initial features equally
supports a readout explanation. A repeated trained-over-initial gain
supports useful pretraining. If neither gains, inspect timing preparation
and the objective before broad mask mixtures. This cohort has already
informed the hypotheses, so new results remain development evidence.
Continue with [18](18_motion_information_and_readout.ipynb), using the
same mode, experiment list, fold/seed scope, device and output root.

## 6. Interpretation of the completed training artifacts

**Evidence snapshot: 2026-09-08.** This copy of Notebook 17 has no saved
code outputs. Completion is established by Notebook 18's executions 5–6
and its [grid manifest](artifacts/motion_structured/grids/292443b0fab5339f5da7ca566a85d6172ffc5b64abe5febf2546681a0152ff57/manifest.json), rather than inferred from
this notebook's workload declaration. The review independently checked
the grid, training and evaluation file hashes, and all 125 training
histories contain steps 1 through 1,200. No model was retrained for this
analysis. The conclusions below concern that specific saved grid.

### Step 1: Separate completion from effectiveness

| Verified item | Meaning |
|---|---|
| 25 motion jobs × 3 arms, 25 region jobs × 2 arms | 50 paired jobs and 125 trained encoder runs |
| 1,200 updates in each history | 150,000 arm-specific optimizer updates; 60,000 paired-job update rounds |
| Five folds × seeds 42–46 | New stochastic training per fold/seed, with the same video partitions reused across seeds |
| CUDA BF16 training; FP32 weights, loss reductions and frozen evaluation | One explicitly identified numerical condition |
| 125,000 held-out prediction rows | 625 clips × five seeds × five arms × eight representations, not 125,000 participants |

Notebook 18's completion bars show 25 jobs per experiment. Their height
is an availability check, not a performance score. The grid contains all
625 accepted clips from 93 videos for each arm/representation/seed, and
the review recomputed all 200 pooled R²/MAE rows from saved predictions.

The current configuration defaults to FP32 unless overridden. To inspect
this completed run, use `PRECISION="bf16"`, the CUDA kernel, the same
fold/seed scope and experiment list, and its existing output root in both
notebooks. An FP32 configuration names a different experiment and may
correctly report these jobs as missing. Do not interpret that as loss of
the BF16 results or launch another grid merely to refill this notebook's
empty output cells. The short FP32/BF16 timing pilots do not establish
numerical equivalence or identify the cause of the scientific outcome.

### Step 2: Interpret the training histories

These values were read from the 25 histories per arm and averaged at
their first and final steps. They are endpoint summaries, not evidence
of monotonic convergence between those steps.

| Experiment / arm | Masked prediction loss, step 1 → 1,200 | Total loss, step 1 → 1,200 |
|---|---:|---:|
| Motion / uniform | 15.078 → 0.849 | 16.247 → 1.335 |
| Motion / MAMP | 15.042 → 0.835 | 16.211 → 1.316 |
| Motion / robust mixture | 15.024 → 0.864 | 16.193 → 1.348 |
| Regions / uniform | 15.232 → 0.839 | 16.402 → 1.324 |
| Connected regions | 15.125 → 0.948 | 16.295 → 1.452 |

Optimization substantially reduces its objective. The total includes
`masked_prediction_loss + 0.05 × variance_regularizer`; the latter
contributes roughly 0.48–0.50 at the final averaged step. That scalar
contribution is not a measurement of its gradient influence. MAMP's
slightly lower prediction loss does not make it the best representation:
each arm has a changing teacher, target distribution and feature space.
Longer training cannot be justified solely by comparing these losses.

### Step 3: Ask what learning achieved

The final teacher with temporal summaries reaches mean R² **0.101–0.114**;
the initial encoder under the same summary reaches **0.223**. All five
trained arms have lower R² and higher MAE than initial features in all
five seeds. The teacher generally outperforms the final online encoder
in the averaged table, but it does not recover the initial readout level.

The predictor has nevertheless learned clip correspondence: across the
75 fold/seed/evaluation-mask rows per trained arm, mismatched targets
always yield larger error than matched targets on the same eligible
clips. This can coexist with an inferior endpoint readout. The evidence
is consistent with learning information that is less useful to this
particular summary/ridge/target combination; it does not identify which
component is responsible or prove that movement information was erased.

### Step 4: Spend the next computation on a discriminating test

1. Reuse the saved encoders for a common expanded ridge grid and the
   summary-component ablations in Notebook 18. Those checks do not
   require repeating these 150,000 updates.
2. If the deficit survives, run one source-separated training pilot with
   initialization and prespecified intermediate checkpoints, for example
   steps 0, 100, 300, 600 and 1,200. Reserve validation sources from that
   pilot's encoder training. Track readout, loss, unscaled feature
   variation and correspondence together; do not select a checkpoint
   on the already inspected outer-test scores.
3. Change one candidate cause at a time. A focused regularizer-weight or
   target-objective comparison is interpretable; changing masking,
   precision, augmentation, width and duration together is not. An FP32
   sensitivity pilot is a separate numerical comparison if needed.
4. Expand a revised recipe to all folds/seeds only after the pilot defines
   a fixed testable hypothesis. Report the present unfavorable result
   alongside any later improvement on this development cohort.

**Takeaway.** The full training milestone is complete. It improved the
training objective and predictor correspondence, but it did not improve
the tested laterality readout over initialization. Diagnose that gap
before increasing the model, training duration or mask menu.